# Setup

In [6]:
import os
if os.path.exists('reviews_classifier'):
    !rm -rf reviews_classifier
!git clone https://github.com/Sirius-Siru/reviews_classifier.git

import sys
if '/content/reviews_classifier' not in sys.path:
    sys.path.append('/content/reviews_classifier')

!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

if os.path.isdir('data') and os.path.exists('data/testData.tsv'):
    print("✨ Dữ liệu đã sẵn sàng, không cần tải lại.")
else:
    print("🚀 Đang tải dữ liệu...")
    !kaggle competitions download -c word2vec-nlp-tutorial
    !unzip -o word2vec-nlp-tutorial.zip -d data
    !unzip -o data/labeledTrainData.tsv.zip -d data
    !unzip -o data/unlabeledTrainData.tsv.zip -d data
    !unzip -o data/testData.tsv.zip -d data

    !rm -f word2vec-nlp-tutorial.zip
    !rm -f data/labeledTrainData.tsv.zip
    !rm -f data/unlabeledTrainData.tsv.zip
    !rm -f data/testData.tsv.zip

Cloning into 'reviews_classifier'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
Receiving objects: 100% (11/11), done.
Resolving deltas: 100% (1/1), done.
remote: Total 11 (delta 1), reused 9 (delta 1), pack-reused 0 (from 0)
✨ Dữ liệu đã sẵn sàng, không cần tải lại.


In [ ]:
import pandas as pd
import numpy as np
import csv
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import src.preprocessing as pcs

# Load data

In [20]:
labeledTrain = pd.read_csv('data/labeledTrainData.tsv', sep='\t', quoting=csv.QUOTE_NONE)
unlabeledTrain = pd.read_csv('data/unlabeledTrainData.tsv', sep='\t', quoting=csv.QUOTE_NONE)
test = pd.read_csv('data/testData.tsv', sep='\t', quoting=csv.QUOTE_NONE)
print(labeledTrain.info())
print(unlabeledTrain.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         25000 non-null  object
 1   sentiment  25000 non-null  int64 
 2   review     25000 non-null  object
dtypes: int64(1), object(2)
memory usage: 586.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      50000 non-null  object
 1   review  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


# Preprocessing

In [ ]:
labeledTrain['review'] = pcs.clean(labeledTrain['review'])
unlabeledTrain['review'] = pcs.clean(unlabeledTrain['review'])
test['review'] = pcs.clean(test['review'])

labeled_tokens = pcs.tokenization(labeledTrain['review'])
unlabeled_tokens = pcs.tokenization(unlabeledTrain['review'])
test_tokens = pcs.tokenization(test['review'])

labeled_bigrams = pcs.to_bigrams(labeled_tokens)
unlabeled_bigrams = pcs.to_bigrams(unlabeled_tokens)
test_bigrams = pcs.to_bigrams(test_tokens)